# # SIMILIS — Multi-Task Deep Learning для классификации археологических находок
# ## Полный пайплайн: от загрузки данных до инференса

# ## Импорт библиотек

In [ ]:
import os
import sys
import random
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchvision

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import scipy.ndimage as ndimage

# Настройка для работы с очень большими изображениями
Image.MAX_IMAGE_PIXELS = None

# ## Шаг 1: Инициализация окружения

In [ ]:
print(f"Python version: {sys.version}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")

# Фиксация Seed для воспроизводимости
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything()

# ## Шаг 2: Создание структуры папок

In [ ]:
folders = [
    'data/raw', 'data/interim', 'data/processed',
    'splits', 'artifacts/checkpoints', 'artifacts/preds',
    'artifacts/reports', 'artifacts/figures'
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created: {folder}")

# ## Шаг 3: Загрузка данных

In [ ]:
if not os.path.exists('cu_data'):
    if not os.path.exists('cu_data.zip'):
        print("Файл не найден. Скачивание...")
        print("(замените на ваш способ загрузки)")
    else:
        print("Архив уже скачан.")
        print("Распаковка данных...")
else:
    print("Данные уже распакованы и готовы к работе.")

# ## Шаг 4: Конфигурация эксперимента

In [ ]:
class Config:
    # Данные и пути
    data_root = 'data'
    raw_dir = os.path.join(data_root, 'raw')
    images_dir = os.path.join(raw_dir, 'dataset')
    csv_path = os.path.join(raw_dir, 'selected_by_name_iimk_subset_public.csv')

    # Логирование и чекпоинты
    checkpoint_dir = 'artifacts/checkpoints'
    log_file = os.path.join('artifacts/reports', 'training_log.csv')

    # Железо и Seed
    seed = 42
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    batch_size = 16
    num_workers = 2

    # Архитектура
    backbone = 'resnet50'
    img_size = 448

    # Оптимизация
    lr = 3e-4
    weight_decay = 1e-4
    epochs = 5
    eta_min = 1e-6

    target_cols = ['name_final', 'material_final', 'fragm_final']
    use_class_weights = True
    description_template = "Археологическая находка: {name}, из материала: {material}. По классификации сохранности: {fragm}."

cfg = Config()

# ## Шаг 5: Чтение и первичный анализ CSV

In [ ]:
try:
    df_raw = pd.read_csv(cfg.csv_path, index_col=0)
    print(f"CSV успешно прочитан, строк: {len(df_raw)}")
except Exception as e:
    print(f"Ошибка чтения CSV: {e}")
    print(f"Проверь наличие файла по пути: {cfg.csv_path}")

# Считаем файлы на диске
image_extensions = ('*.jpg', '*.jpeg', '*.png', '*.JPG', '*.PNG')
image_files = []
for ext in image_extensions:
    path_pattern = os.path.join(cfg.images_dir, '**', ext)
    image_files.extend(glob.glob(path_pattern, recursive=True))

print(f"Количество изображений в {cfg.images_dir}: {len(image_files)}")

if len(df_raw) != len(image_files):
    print("Количество записей в CSV и файлов на диске не совпадает!")

# ## Шаг 6: Маппинг изображений и проверка целостности

In [ ]:
df = df_raw.copy()
df['image_file'] = df['code'].apply(lambda x: f"{x}_orig.jpg")
group_key = 'code'

df['is_valid'] = df['image_file'].apply(
    lambda x: os.path.exists(os.path.join(cfg.images_dir, x))
)

valid_count = df['is_valid'].sum()
broken_count = len(df) - valid_count
dup_images = df['image_file'].duplicated().sum()
dup_groups = df[group_key].duplicated().sum()

print(f"Колонки: {df.columns.tolist()}")
print(f"Валидных путей: {valid_count} | Битых путей: {broken_count}")
print(f"Дубликатов файлов: {dup_images} | Повторов ID: {dup_groups}")

# Очистка от битых ссылок
df = df[df['is_valid']].copy()

# ## Шаг 7: Визуализация примеров данных

In [ ]:
if not df.empty:
    n_samples = 3
    samples = df.sample(n_samples, random_state=42)

    for idx, row in samples.iterrows():
        plt.figure(figsize=(12, 4))
        img_path = os.path.join(cfg.images_dir, row['image_file'])
        try:
            img = Image.open(img_path)
            plt.subplot(1, 2, 1)
            plt.imshow(img)
            plt.axis('off')
            plt.title(f"File: {row['image_file']}")
        except:
            print(f"Не удалось открыть {img_path}")

        plt.subplot(1, 2, 2)
        meta_info = [f"{col}: {row[col]}" for col in ['code', 'name', 'material', 'fragm'] if col in row]
        meta_text = "\n".join(meta_info)
        plt.text(0.1, 0.5, meta_text, fontsize=11, va='center', family='monospace')
        plt.axis('off')
        plt.title("Metadata")
        plt.tight_layout()
        plt.show()

# Сохранение маппинга
output_path = 'data/interim/inventory_mapped.csv'
df.to_csv(output_path, index=False)
print(f"Валидный датасет сохранен в: {output_path}")

# ## Шаг 8: Сбор геометрических характеристик изображений

In [ ]:
df = pd.read_csv('data/interim/inventory_mapped.csv')
IMAGES_DIR = cfg.images_dir

widths, heights, aspect_ratios = [], [], []
for img_name in tqdm(df['image_file'], desc="Checking image sizes"):
    with Image.open(os.path.join(IMAGES_DIR, img_name)) as img:
        w, h = img.size
        widths.append(w)
        heights.append(h)
        aspect_ratios.append(w / h)

df['width'], df['height'], df['aspect_ratio'] = widths, heights, aspect_ratios
df = df[df['is_valid']].copy()

# ## Шаг 9: Визуализация распределений

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.histplot(df['width'], bins=30, ax=axes[0], color='skyblue').set_title('Distribution of Widths')
sns.histplot(df['aspect_ratio'], bins=30, ax=axes[1], color='salmon').set_title('Aspect Ratio (W/H)')
df['code'].value_counts().value_counts().plot(kind='bar', ax=axes[2], color='lightgreen')
axes[2].set_title('Images per Artifact (Group Size)')
axes[2].set_xlabel('Num Images'), axes[2].set_ylabel('Count')
plt.tight_layout()
plt.show()

# ## Шаг 10: Сетка примеров изображений

In [ ]:
def show_grid(df, n=12, cols=4):
    rows = n // cols
    plt.figure(figsize=(20, 5 * rows))
    samples = df.sample(n, random_state=42)
    for i, (_, row) in enumerate(samples.iterrows()):
        plt.subplot(rows, cols, i + 1)
        img = Image.open(os.path.join(IMAGES_DIR, row['image_file']))
        plt.imshow(img)
        label = f"{row['name'][:20]}\n{row['material']} | {row['fragm']}\n{img.size[0]}x{img.size[1]}"
        plt.title(label, fontsize=10)
        plt.axis('off')
    plt.tight_layout()
    plt.show()

show_grid(df)

# ## Шаг 11: Визуальный эвристический анализ

In [ ]:
def advanced_visual_heuristic(img_path):
    with Image.open(img_path).convert('L') as img:
        img_np = np.array(img.resize((256, 256)))
        corners = [img_np[0:20, 0:20], img_np[0:20, -20:], img_np[-20:, 0:20], img_np[-20:, -20:]]
        bg_mean = np.mean(corners)
        if bg_mean > 220: bg_type = 'white'
        elif bg_mean > 150: bg_type = 'light-gray'
        else: bg_type = 'other/dark'

        fg_mask = np.abs(img_np - bg_mean) > 30
        foreground_ratio = np.sum(fg_mask) / img_np.size

        grad = ndimage.gaussian_gradient_magnitude(img_np.astype(float), sigma=1)
        bottom_strip = grad[200:, :]
        has_scale_bar = np.mean(bottom_strip) > 5
        has_overlay_text = np.max(grad) > 50

        center_mass = np.sum(fg_mask[64:192, 64:192]) / np.sum(fg_mask + 1e-6)
        if center_mass < 0.4:
            layout_mode = 'multi-view/complex'
        elif foreground_ratio < 0.1:
            layout_mode = 'close-up/macro'
        else:
            layout_mode = 'single_fragment'

    return bg_type, foreground_ratio, has_scale_bar, has_overlay_text, layout_mode

# Применяем на подвыборке
subset_indices = df.sample(50, random_state=42).index
results = [advanced_visual_heuristic(os.path.join(IMAGES_DIR, df.loc[i, 'image_file'])) for i in subset_indices]

df_sub = df.loc[subset_indices].copy()
df_sub['bg_type'], df_sub['fg_ratio'], df_sub['has_scale_bar'], \
df_sub['has_overlay_text'], df_sub['layout_mode'] = zip(*results)

print("\nВизуальный анализ (n=50)")
print(f"Типы фона: {df_sub['bg_type'].value_counts(normalize=True)}")
print(f"Режимы раскладки: {df_sub['layout_mode'].value_counts()}")
print(f"Наличие линейки: {df_sub['has_scale_bar'].sum()} шт.")
print(f"Наличие текста: {df_sub['has_overlay_text'].sum()} шт.")
print(f"Средняя заполненность кадра: {df_sub['fg_ratio'].mean():.2f}")

# ## Шаг 12: Анализ проблемных примеров

In [ ]:
problem_indices = [6, 8, 9, 12, 18]
plt.figure(figsize=(20, 10))
for i, idx in enumerate(problem_indices):
    plt.subplot(1, len(problem_indices), i + 1)
    if idx in df.index:
        row = df.loc[idx]
        img_path = os.path.join(IMAGES_DIR, row['image_file'])
        try:
            img = Image.open(img_path)
            plt.imshow(img)
            info = f"ID: {idx}\nSize: {img.size[0]}x{img.size[1]}\nAspect: {row['aspect_ratio']:.2f}"
            plt.title(info, fontsize=12, fontweight='bold')
        except Exception as e:
            plt.title(f"ID: {idx} - Error")
            print(f"Не удалось открыть изображение по индексу {idx}: {e}")
    else:
        plt.title(f"ID: {idx} - Not in df")
        plt.text(0.5, 0.5, "Index missing", ha='center')
    plt.axis('off')
plt.tight_layout()
plt.show()

# ## Шаг 13: Текстовый анализ полей

In [ ]:
pd.set_option('display.max_colwidth', None)
print(df[['code', 'name', 'description', 'material', 'fragm']].sample(20, random_state=42))

selected_fields = ['name', 'material', 'fragm']
print("Анализ выбранных полей")
for field in selected_fields:
    unique_vals = df[field].unique()
    print(f"\nПоле: {field}")
    print(f"Уникальных значений: {len(unique_vals)}")
    print(f"Примеры: {unique_vals[:10]}")
    missing = df[field].isna().mean() * 100
    print(f"Доля пропусков: {missing:.2f}%")

# ## Шаг 14: Нормализация текста и маппинг синонимов

In [ ]:
def normalize_text(text):
    if pd.isna(text): return "unknown"
    text = str(text).lower().strip()
    text = text.replace("фр-т", "фрагмент").replace(" ф-т", " фрагмент")
    return text

df['name_norm'] = df['name'].apply(normalize_text)
df['material_norm'] = df['material'].apply(normalize_text)

name_mapping = {
    "изразца": "изразец", "изразца фрагмент": "изразец", "фр-т изразца": "изразец",
    "гвоздя": "гвоздь", "гвоздя фрагмент": "гвоздь",
    "сосуда": "сосуд", "фрагмент сосуда": "сосуд", "стенка сосуда": "сосуд"
}
material_mapping = {
    "керамика.": "керамика", "чернолощеная керамика": "керамика",
    "белоглиняная керамика": "керамика", "железо": "металл", "цветной металл": "металл"
}

df['name_norm'] = df['name_norm'].replace(name_mapping)
df['material_norm'] = df['material_norm'].replace(material_mapping)

# ## Шаг 15: Финализация меток (редкие классы → other)

In [ ]:
def finalize_labels(df, col, min_samples=5):
    df[f'{col}_is_missing'] = df[col].isna()
    counts = df[f'{col}_norm'].value_counts()
    rare_classes = counts[counts < min_samples].index
    df[f'{col}_final'] = df[f'{col}_norm'].replace(rare_classes, 'other')
    df[f'{col}_is_uncertain'] = df[col].astype(str).str.contains(r'\?', na=False)
    return df

for col in ['name', 'material']:
    df = finalize_labels(df, col)

df['fragm_final'] = df['fragm'].apply(normalize_text).replace({'фрагмент': 'fragment', 'целое': 'whole'})

print("Распределение TOP-10 классов (Name)")
print(df['name_final'].value_counts().head(10))
print(f"\nЗаменено на 'other' (Name): {len(df[df['name_final'] == 'other'])}")
print(f"Неоднозначных записей (?): {df['name_is_uncertain'].sum()}")

# ## Шаг 16: Генератор текстового описания (auto_description)

In [ ]:
def auto_description_engine(preds, confs, thresholds={'high': 0.85, 'med': 0.6}):
    """
    Генератор описания на основе предсказаний мультитаск-модели.
    Args:
        preds: словарь с предсказанными метками категорий
        confs: словарь с уверенностью модели (0.0 - 1.0) для каждой категории
    """
    parts = []
    name_val = preds.get('name', 'артефакт').lower()
    name_p = confs.get('name', 0)
    if name_p > thresholds['high']:
        parts.append(name_val.capitalize())
    elif name_p > thresholds['med']:
        parts.append(f"Предположительно {name_val}")
    else:
        parts.append("Неопределенный артефакт")

    mat_val = preds.get('material', '').lower()
    mat_p = confs.get('material', 0)
    if mat_p > thresholds['high']:
        parts.append(f"из материала: {mat_val}")
    elif mat_p > thresholds['med']:
        parts.append(f"(возм. {mat_val})")

    frag_val = preds.get('fragm', '').lower()
    frag_p = confs.get('fragm', 0)
    if frag_p > thresholds['high']:
        if 'фрагмент' in frag_val or 'fragment' in frag_val:
            parts.append(", в состоянии фрагмента")
        elif 'целое' in frag_val or 'whole' in frag_val:
            parts.append(", целой сохранности")

    res = " ".join(parts).replace(" ,", ",")
    return res.strip() + "."

# Тестирование генератора
test_scenarios = [
    {"desc": "Идеальное предсказание (High Confidence)",
     "preds": {"name": "изразец", "material": "керамика", "fragm": "фрагмент"},
     "confs": {"name": 0.98, "material": 0.95, "fragm": 0.92}},
    {"desc": "Сомнение в типе предмета (Medium Confidence)",
     "preds": {"name": "гвоздь", "material": "металл", "fragm": "целое"},
     "confs": {"name": 0.65, "material": 0.91, "fragm": 0.88}},
    {"desc": "Низкая уверенность в материале",
     "preds": {"name": "сосуд", "material": "стекло", "fragm": "фрагмент"},
     "confs": {"name": 0.96, "material": 0.35, "fragm": 0.90}},
    {"desc": "Слабая уверенность во всём (Fallback)",
     "preds": {"name": "монета", "material": "металл", "fragm": "целое"},
     "confs": {"name": 0.45, "material": 0.40, "fragm": 0.30}},
    {"desc": "Средняя уверенность в материале",
     "preds": {"name": "бусина", "material": "янтарь", "fragm": "целое"},
     "confs": {"name": 0.92, "material": 0.70, "fragm": 0.89}}
]

for ts in test_scenarios:
    result = auto_description_engine(ts['preds'], ts['confs'])
    print(f"{ts['desc']:<40} | {result}")

# ## Шаг 17: Разбиение на Train/Val/Test (GroupShuffleSplit)

In [ ]:
gss_test = GroupShuffleSplit(n_splits=1, train_size=0.85, random_state=42)
train_val_idx, test_open_idx = next(gss_test.split(df, groups=df['code']))
df_train_val = df.iloc[train_val_idx].reset_index(drop=True)
df_test = df.iloc[test_open_idx].reset_index(drop=True)

gss_val = GroupShuffleSplit(n_splits=1, train_size=0.824, random_state=42)
train_idx, val_idx = next(gss_val.split(df_train_val, groups=df_train_val['code']))
df_train = df_train_val.iloc[train_idx].reset_index(drop=True)
df_val = df_train_val.iloc[val_idx].reset_index(drop=True)

print(f"Размер train: {len(df_train)} | val: {len(df_val)} | test: {len(df_test)}")

# ## Шаг 18: Проверка сплитов на утечки данных

In [ ]:
def check_splits(train, val, test):
    print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
    set_train = set(train['code'])
    set_val = set(val['code'])
    set_test = set(test['code'])
    inter_train_val = set_train.intersection(set_val)
    inter_val_test = set_val.intersection(set_test)
    inter_train_test = set_train.intersection(set_test)
    print(f"Пересечение Train/Val: {len(inter_train_val)} групп")
    print(f"Пересечение Val/Test: {len(inter_val_test)} групп")
    print(f"Пересечение Train/Test: {len(inter_train_test)} групп")
    if len(inter_train_val) + len(inter_val_test) + len(inter_train_test) == 0:
        print("Утечек по группам не обнаружено.")
    else:
        print("Обнаружены общие группы в разных сплитах!")

check_splits(df_train, df_val, df_test)

# ## Шаг 19: Визуализация распределения классов по сплитам

In [ ]:
plt.figure(figsize=(15, 5))
target_col = 'material_final'
for i, (name, d) in enumerate([('Train', df_train), ('Val', df_val), ('Test', df_test)]):
    plt.subplot(1, 3, i+1)
    d[target_col].value_counts(normalize=True).head(5).plot(kind='bar', color='orange')
    plt.title(f"Dist in {name}")
    plt.ylim(0, 0.8)
plt.tight_layout()
plt.show()

# ## Шаг 20: Определение аугментаций (Albumentations)

In [ ]:
SIZE = 448

# Пайплайн для тренировки (с аугментациями)
train_transform = A.Compose([
    A.LongestMaxSize(max_size=SIZE),
    A.PadIfNeeded(min_height=SIZE, min_width=SIZE, border_mode=cv2.BORDER_CONSTANT, fill=0),
    A.HorizontalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(
        translate_percent={"x": (-0.05, 0.05), "y": (-0.05, 0.05)},
        scale=(0.95, 1.05), rotate=(-15, 15),
        interpolation=cv2.INTER_LINEAR, border_mode=cv2.BORDER_CONSTANT, fill=0, p=0.5
    ),
    A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# Пайплайн для валидации (только resize+pad+нормализация)
val_transform = A.Compose([
    A.LongestMaxSize(max_size=SIZE),
    A.PadIfNeeded(min_height=SIZE, min_width=SIZE, border_mode=cv2.BORDER_CONSTANT, fill=0),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# ## Шаг 21: Визуализация аугментаций

In [ ]:
def visualize_augmentations(df, transform, n=3):
    sample = df.sample(1).iloc[0]
    img_path = os.path.join(IMAGES_DIR, sample['image_file'])
    image = cv2.imread(img_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(15, 5))
    plt.subplot(1, n+1, 1)
    plt.imshow(image)
    plt.title("Original")
    plt.axis('off')
    for i in range(n):
        augmented = transform(image=image)['image']
        aug_img = augmented.permute(1, 2, 0).numpy()
        aug_img = (aug_img * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
        aug_img = np.clip(aug_img, 0, 1)
        plt.subplot(1, n+1, i+2)
        plt.imshow(aug_img)
        plt.title(f"Train Aug {i+1}")
        plt.axis('off')
    plt.show()

print("Проверка Train Аугментаций")
visualize_augmentations(df_train, train_transform)
print("Проверка Val Трансформаций")
visualize_augmentations(df_val, val_transform, n=1)

# ## Шаг 22: Кодирование меток (LabelEncoder)

In [ ]:
le_dict = {}
target_cols = ['name_final', 'material_final', 'fragm_final']

for col in target_cols:
    le = LabelEncoder()
    train_values = df_train[col].astype(str).unique().tolist()
    if 'unknown' not in train_values:
        train_values.append('unknown')
    le.fit(train_values)
    le_dict[col] = le

    df_train[col + '_encoded'] = le.transform(df_train[col].astype(str))
    df_val[col + '_encoded'] = df_val[col].astype(str).apply(lambda x: x if x in le.classes_ else 'unknown')
    df_val[col + '_encoded'] = le.transform(df_val[col + '_encoded'])
    df_test[col + '_encoded'] = df_test[col].astype(str).apply(lambda x: x if x in le.classes_ else 'unknown')
    df_test[col + '_encoded'] = le.transform(df_test[col + '_encoded'])

    print(f"Энкодер для {col} готов. Классов: {len(le.classes_)}")

# ## Шаг 23: Создание Dataset и DataLoader

In [ ]:
class SimilisDataset(Dataset):
    """Датасет для мультитаск-классификации археологических находок."""
    def __init__(self, df, image_dir, transform=None, label_encoders=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform
        self.label_encoders = label_encoders
        self.target_cols = ['name_final', 'material_final', 'fragm_final']

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_dir, row['image_file'])
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)['image']

        targets = {}
        target_masks = {}
        for col in self.target_cols:
            val = row[col]
            if val == 'unknown' or pd.isna(val):
                targets[col] = torch.tensor(0, dtype=torch.long)
                target_masks[col] = torch.tensor(0.0, dtype=torch.float32)
            else:
                idx_val = self.label_encoders[col].transform([val])[0]
                targets[col] = torch.tensor(idx_val, dtype=torch.long)
                target_masks[col] = torch.tensor(1.0, dtype=torch.float32)

        metadata = {'image_file': row['image_file'], 'group_key': str(row['code'])}
        return {'image': image, 'targets': targets, 'target_masks': target_masks, 'metadata': metadata}

# Сборка лоадеров
train_dataset = SimilisDataset(df_train, IMAGES_DIR, transform=train_transform, label_encoders=le_dict)
val_dataset = SimilisDataset(df_val, IMAGES_DIR, transform=val_transform, label_encoders=le_dict)
test_dataset = SimilisDataset(df_test, IMAGES_DIR, transform=val_transform, label_encoders=le_dict)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Проверка batch
batch = next(iter(train_loader))
print(f"Image shape: {batch['image'].shape}")
print(f"Target 'name' shape: {batch['targets']['name_final'].shape}")

# ## Шаг 24: Sanity Check датасета

In [ ]:
def run_sanity_check(dataset, idx=0):
    print(f"Sanity Check для элемента #{idx}")
    item = dataset[idx]
    print(f"Файл: {item['metadata']['image_file']}")
    print(f"Группа (code): {item['metadata']['group_key']}")
    decoded_labels = {}
    for col in ['name_final', 'material_final', 'fragm_final']:
        idx_val = item['targets'][col].item()
        label = le_dict[col].inverse_transform([idx_val])[0]
        mask = item['target_masks'][col].item()
        decoded_labels[col] = label if mask > 0 else "MISSING"
    print(f"Декодированные метки: {decoded_labels}")
    fake_confs = {k.split('_')[0]: 1.0 for k in decoded_labels.keys()}
    fake_preds = {k.split('_')[0]: v for k, v in decoded_labels.items()}
    description = auto_description_engine(fake_preds, fake_confs)
    print(f"Auto-Description: {description}")
    img_display = item['image'].permute(1, 2, 0).numpy()
    img_display = (img_display * [0.229, 0.224, 0.225]) + [0.485, 0.456, 0.406]
    plt.imshow(np.clip(img_display, 0, 1))
    plt.title(f"Check: {item['metadata']['image_file']}")
    plt.show()

run_sanity_check(val_dataset, idx=5)

# ## Шаг 25: Определение архитектуры модели (Multi-Task ResNet50)

In [ ]:
class SimilisMultiTaskModel(nn.Module):
    """Multi-Task модель на базе ResNet50 для классификации археологических находок."""
    def __init__(self, num_classes_dict, backbone_name='resnet50', pretrained=True):
        super(SimilisMultiTaskModel, self).__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.num_features = self.backbone.num_features
        self.heads = nn.ModuleDict({
            'name': nn.Linear(self.num_features, num_classes_dict['name_final']),
            'material': nn.Linear(self.num_features, num_classes_dict['material_final']),
            'fragm': nn.Linear(self.num_features, num_classes_dict['fragm_final'])
        })

    def forward(self, x):
        embedding = self.backbone(x)  # (B, 2048)
        logits = {
            'name': self.heads['name'](embedding),
            'material': self.heads['material'](embedding),
            'fragm': self.heads['fragm'](embedding)
        }
        return logits

# Подготовка словаря количества классов
num_classes_dict = {
    'name_final': len(le_dict['name_final'].classes_),
    'material_final': len(le_dict['material_final'].classes_),
    'fragm_final': len(le_dict['fragm_final'].classes_)
}

model = SimilisMultiTaskModel(num_classes_dict)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Backbone: ResNet50")
print(f"Обучаемых параметров: {count_parameters(model):,}")
for task, num in num_classes_dict.items():
    print(f"{task}_logits: (B, {num})")

# ## Шаг 26: Проверка Forward Pass

In [ ]:
batch = next(iter(train_loader))
images = batch['image']
model.eval()
with torch.no_grad():
    outputs = model(images)

print("Проверка Forward Pass")
for task, logits in outputs.items():
    print(f"Задача {task:10}: логиты {tuple(logits.shape)}")

if all(outputs[t].shape[0] == images.shape[0] for t in outputs):
    print("[OK] Forward pass прошел успешно.")

# ## Шаг 27: Функция вычисления метрик батча

In [ ]:
def compute_batch_metrics(logits_dict, targets_dict, masks_dict, weights=None):
    """
    Вычисление loss и accuracy для мультитаск-модели.
    Применяет маски для игнорирования unknown-меток и взвешивание классов.
    """
    total_loss = 0
    results = {}

    for task in logits_dict.keys():
        target_key = f"{task}_final"
        logits = logits_dict[task]
        target = targets_dict[target_key]
        mask = masks_dict[target_key]

        task_weights = weights[task] if weights is not None and task in weights else None
        raw_loss = F.cross_entropy(logits, target, weight=task_weights, reduction='none')
        masked_loss = raw_loss * mask
        num_valid = mask.sum()
        task_loss = masked_loss.sum() / (num_valid + 1e-8)
        total_loss += task_loss

        with torch.no_grad():
            preds = torch.argmax(logits, dim=1)
            correct = (preds == target).float() * mask
            task_acc = correct.sum() / (num_valid + 1e-8)

        results[task] = {'loss': task_loss.item(), 'acc': task_acc.item()}

    return total_loss, results

# ## Шаг 28: Overfit-тест (проверка способности модели обучаться)

In [ ]:
tiny_indices = list(range(32))
tiny_subset = torch.utils.data.Subset(train_dataset, tiny_indices)
tiny_subset.dataset.transform = val_transform
tiny_loader = DataLoader(tiny_subset, batch_size=16, shuffle=False)
print(f"Размер tiny-subset: {len(tiny_subset)} изображений")

model.to(cfg.device)
optimizer_overfit = optim.Adam(model.parameters(), lr=1e-4)
history = {'loss': [], 'acc_name': [], 'acc_material': []}
epochs_overfit = 50

pbar_epochs = tqdm(range(epochs_overfit), desc="Overfit Training")
for epoch in pbar_epochs:
    model.train()
    epoch_loss = 0
    correct_name = 0
    correct_mat = 0
    for batch in tiny_loader:
        images = batch['image'].to(cfg.device)
        targets = {k: v.to(cfg.device) for k, v in batch['targets'].items()}
        masks = {k: v.to(cfg.device) for k, v in batch['target_masks'].items()}
        optimizer_overfit.zero_grad()
        logits = model(images)
        loss, metrics = compute_batch_metrics(logits, targets, masks)
        loss.backward()
        optimizer_overfit.step()
        epoch_loss += loss.item()
        with torch.no_grad():
            correct_name += (torch.argmax(logits['name'], dim=1) == targets['name_final']).float().sum().item()
            correct_mat += (torch.argmax(logits['material'], dim=1) == targets['material_final']).float().sum().item()

    current_loss = epoch_loss / len(tiny_loader)
    current_acc_n = correct_name / len(tiny_subset)
    current_acc_m = correct_mat / len(tiny_subset)
    history['loss'].append(current_loss)
    history['acc_name'].append(current_acc_n)
    history['acc_material'].append(current_acc_m)
    pbar_epochs.set_postfix({'loss': f"{current_loss:.4f}", 'acc_name': f"{current_acc_n:.2%}", 'acc_mat': f"{current_acc_m:.2%}"})

# ## Шаг 29: Визуализация кривых обучения (Overfit Test)

In [ ]:
def plot_training_history(history, title_prefix="Overfit Test"):
    """Отрисовка кривых loss и accuracy по истории обучения."""
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(history['loss'], label='Train Loss', color='red')
    plt.title(f'Кривая Train Loss ({title_prefix})')
    plt.xlabel('Эпоха')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.subplot(1, 2, 2)
    plt.plot(history['acc_name'], label='Accuracy Name')
    plt.plot(history['acc_material'], label='Accuracy Material')
    plt.title(f'Кривые Accuracy ({title_prefix})')
    plt.xlabel('Эпоха')
    plt.legend()
    plt.grid(True)
    plt.show()

plot_training_history(history, "Overfit Test")

# ## Шаг 30: Вспомогательные функции (train_one_epoch, evaluate, get_class_weights)

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    """Одна эпоха обучения."""
    model.train()
    total_loss = 0
    pbar = tqdm(loader, desc="Training", leave=False)
    for batch in pbar:
        images = batch['image'].to(device)
        targets = {k: v.to(device) for k, v in batch['targets'].items()}
        masks = {k: v.to(device) for k, v in batch['target_masks'].items()}
        optimizer.zero_grad()
        logits_dict = model(images)
        loss, _ = compute_batch_metrics(logits_dict, targets, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, device, le_dict):
    """Оценка модели на валидационной выборке."""
    model.eval()
    total_loss = 0
    y_true = {task: [] for task in ['name', 'material', 'fragm']}
    y_pred = {task: [] for task in ['name', 'material', 'fragm']}
    y_mask = {task: [] for task in ['name', 'material', 'fragm']}

    for batch in tqdm(loader, desc="Evaluating", leave=False):
        images = batch['image'].to(device)
        targets = {k: v.to(device) for k, v in batch['targets'].items()}
        masks = {k: v.to(device) for k, v in batch['target_masks'].items()}
        logits_dict = model(images)
        loss, _ = compute_batch_metrics(logits_dict, targets, masks)
        total_loss += loss.item()
        for task in ['name', 'material', 'fragm']:
            t_key = f"{task}_final"
            preds = torch.argmax(logits_dict[task], dim=1).cpu().numpy()
            y_pred[task].extend(preds)
            y_true[task].extend(targets[t_key].cpu().numpy())
            y_mask[task].extend(masks[t_key].cpu().numpy())

    final_metrics = {'loss': total_loss / len(loader)}
    f1_scores = []
    for task in ['name', 'material', 'fragm']:
        mask = np.array(y_mask[task]) > 0
        valid_true = np.array(y_true[task])[mask]
        valid_pred = np.array(y_pred[task])[mask]
        acc = accuracy_score(valid_true, valid_pred)
        f1 = f1_score(valid_true, valid_pred, average='macro')
        final_metrics[f'{task}_acc'] = acc
        final_metrics[f'{task}_f1'] = f1
        f1_scores.append(f1)
    final_metrics['avg_f1'] = np.mean(f1_scores)
    return final_metrics

def get_class_weights(df, col, le):
    """Вычисление весов классов (обратная частота) для борьбы с дисбалансом."""
    counts = df[col].value_counts().to_dict()
    classes = le.classes_
    weights = []
    for cls in classes:
        count = counts.get(cls, 1)
        weights.append(1.0 / count)
    weights = torch.tensor(weights, dtype=torch.float32)
    weights = weights / weights.sum() * len(classes)
    return weights

# ## Шаг 31: Вычисление весов классов

In [ ]:
weights_dict = {
    'name': get_class_weights(df_train, 'name_final', le_dict['name_final']).to(cfg.device),
    'material': get_class_weights(df_train, 'material_final', le_dict['material_final']).to(cfg.device),
    'fragm': get_class_weights(df_train, 'fragm_final', le_dict['fragm_final']).to(cfg.device)
}

# ## Шаг 32: Инициализация оптимизатора и scheduler

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.eta_min)

# ## Шаг 33: Функции сохранения чекпоинтов и логирования

In [ ]:
def save_checkpoint(model, optimizer, scheduler, epoch, best_metric, is_best=False):
    """Сохранение чекпоинта модели со всеми метаданными."""
    label_mapping = {col: le_dict[col].classes_.tolist() for col in cfg.target_cols}
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'best_metric': best_metric,
        'label_mapping': label_mapping,
        'target_cols': cfg.target_cols,
        'description_template': cfg.description_template
    }
    last_path = os.path.join(cfg.checkpoint_dir, 'last.pt')
    torch.save(checkpoint, last_path)
    if is_best:
        best_path = os.path.join(cfg.checkpoint_dir, 'best.pt')
        torch.save(checkpoint, best_path)
        print(f"Saved BEST checkpoint at epoch {epoch+1}")

def log_to_csv(metrics_dict, epoch, path):
    """Логирование метрик эпохи в CSV."""
    df = pd.DataFrame([metrics_dict])
    df['epoch'] = epoch
    if not os.path.isfile(path):
        df.to_csv(path, index=False)
    else:
        df.to_csv(path, mode='a', header=False, index=False)

def verify_checkpoint(path, sample_batch):
    """Проверка целостности сохранённого чекпоинта."""
    print(f"Тест восстановления из {path}")
    checkpoint = torch.load(path)
    test_model = SimilisMultiTaskModel(num_classes_dict).to(cfg.device)
    test_model.load_state_dict(checkpoint['model_state_dict'])
    test_model.eval()
    with torch.no_grad():
        img = sample_batch['image'][0:1].to(cfg.device)
        orig_output = model(img)
        restored_output = test_model(img)
        diff = torch.abs(orig_output['name'] - restored_output['name']).sum().item()
    if diff < 1e-5:
        print(f"Успех! Разница логитов: {diff:.6f}")
        print(f"Лучшая метрика (Avg F1): {checkpoint['best_metric']:.4f}")
    else:
        print(f"Ошибка! Модели выдают разные результаты.")

# ## Шаг 34: Основной цикл обучения

In [ ]:
# Сброс оптимизатора и scheduler для основного цикла
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
scheduler = CosineAnnealingLR(optimizer, T_max=cfg.epochs, eta_min=cfg.eta_min)
model.to(cfg.device)
best_f1 = 0.0

for epoch in range(cfg.epochs):
    print(f"\nЭпоха {epoch+1}/{cfg.epochs}")

    # ТРЕНИРОВКА
    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Train Ep {epoch+1}")
    for batch in pbar:
        images = batch['image'].to(cfg.device)
        targets = {k: v.to(cfg.device) for k, v in batch['targets'].items()}
        masks = {k: v.to(cfg.device) for k, v in batch['target_masks'].items()}
        optimizer.zero_grad()
        logits_dict = model(images)
        loss, _ = compute_batch_metrics(logits_dict, targets, masks, weights=weights_dict)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}", 'lr': f"{optimizer.param_groups[0]['lr']:.2e}"})

    avg_train_loss = train_loss / len(train_loader)

    # ВАЛИДАЦИЯ
    val_metrics = evaluate(model, val_loader, cfg.device, le_dict)
    val_metrics['train_loss'] = avg_train_loss

    # ЛОГИРОВАНИЕ И ЧЕКПОИНТЫ
    log_to_csv(val_metrics, epoch, cfg.log_file)
    is_best = val_metrics['avg_f1'] > best_f1
    if is_best:
        best_f1 = val_metrics['avg_f1']
    save_checkpoint(model, optimizer, scheduler, epoch, best_f1, is_best=is_best)
    scheduler.step()

    # ПЕЧАТЬ ИТОГОВ ЭПОХИ
    print(f"Эпоха {epoch+1} завершена!")
    print(f"Loss: Train {avg_train_loss:.4f} | Val {val_metrics['loss']:.4f}")
    print(f"F1: Name {val_metrics['name_f1']:.4f} | Mat {val_metrics['material_f1']:.4f} | Avg {val_metrics['avg_f1']:.4f}")

# ## Шаг 35: Загрузка лучшей модели и инференс на val

In [ ]:
best_model_path = os.path.join(cfg.checkpoint_dir, 'best.pt')
print(f"Загрузка лучшего чекпоинта из {best_model_path}...")
checkpoint = torch.load(best_model_path, weights_only=False)

eval_model = SimilisMultiTaskModel(num_classes_dict).to(cfg.device)
eval_model.load_state_dict(checkpoint['model_state_dict'])
eval_model.eval()

# Сбор предсказаний
y_true_all = {task: [] for task in ['name', 'material', 'fragm']}
y_pred_all = {task: [] for task in ['name', 'material', 'fragm']}
y_mask_all = {task: [] for task in ['name', 'material', 'fragm']}

meta_factors = {'layout_mode': [], 'bg_type': [], 'foreground_ratio': [], 'has_scale': []}

with torch.no_grad():
    for batch in val_loader:
        images = batch['image'].to(cfg.device)
        targets = {k: v.to(cfg.device) for k, v in batch['targets'].items()}
        masks = {k: v.to(cfg.device) for k, v in batch['target_masks'].items()}
        logits_dict = eval_model(images)
        for task in ['name', 'material', 'fragm']:
            t_key = f"{task}_final"
            preds = torch.argmax(logits_dict[task], dim=1).cpu().numpy()
            y_pred_all[task].extend(preds)
            y_true_all[task].extend(targets[t_key].cpu().numpy())
            y_mask_all[task].extend(masks[t_key].cpu().numpy())
        for factor in meta_factors.keys():
            if factor in batch:
                meta_factors[factor].extend(batch[factor])
            else:
                meta_factors[factor].extend(['undefined'] * len(images))

# ## Шаг 36: Итоговая таблица метрик

In [ ]:
metrics_rows = []
for task in ['name', 'material', 'fragm']:
    mask = np.array(y_mask_all[task]) > 0
    valid_true = np.array(y_true_all[task])[mask]
    valid_pred = np.array(y_pred_all[task])[mask]
    acc = accuracy_score(valid_true, valid_pred)
    f1_macro = f1_score(valid_true, valid_pred, average='macro')
    metrics_rows.append({'Поле (Task)': task.upper(), 'Accuracy': f"{acc:.2%}", 'Macro-F1': f"{f1_macro:.4f}"})

df_metrics = pd.DataFrame(metrics_rows)
print(df_metrics.to_string(index=False))

# ## Шаг 37: Confusion Matrix для Name и Material

In [ ]:
for task in ['name', 'material']:
    mask = np.array(y_mask_all[task]) > 0
    valid_true = np.array(y_true_all[task])[mask]
    valid_pred = np.array(y_pred_all[task])[mask]
    le = le_dict[f"{task}_final"]
    class_names = le.classes_
    cm = confusion_matrix(valid_true, valid_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(f'Confusion Matrix для поля: {task.upper()}', fontsize=14)
    plt.ylabel('Истинные классы', fontsize=12)
    plt.xlabel('Предсказанные классы', fontsize=12)
    plt.xticks(rotation=90)
    plt.tight_layout()
    plt.show()

# ## Шаг 38: Анализ лучших/худших классов

In [ ]:
target_analysis_task = 'material'
mask = np.array(y_mask_all[target_analysis_task]) > 0
valid_true = np.array(y_true_all[target_analysis_task])[mask]
valid_pred = np.array(y_pred_all[target_analysis_task])[mask]
le = le_dict[f"{target_analysis_task}_final"]
present_classes = np.unique(np.concatenate([valid_true, valid_pred]))
present_class_names = [le.classes_[idx] for idx in present_classes]

report = classification_report(valid_true, valid_pred, labels=present_classes,
                                target_names=present_class_names, output_dict=True, zero_division=0)
df_report = pd.DataFrame(report).transpose().iloc[:-3]
df_report = df_report.sort_values(by='f1-score', ascending=False)

print(f"5 ЛУЧШИХ КЛАССОВ ПО ПОЛЮ {target_analysis_task.upper()}")
print(df_report.head(5)[['f1-score', 'support']].to_string())
print(f"\n5 ХУДШИХ КЛАССОВ ПО ПОЛЮ {target_analysis_task.upper()}")
print(df_report.tail(5)[['f1-score', 'support']].to_string())

# ## Шаг 39: Breakdown анализ ошибок

In [ ]:
analysis_task = 'name'
mask = np.array(y_mask_all[analysis_task]) > 0
is_correct = np.array(y_true_all[analysis_task]) == np.array(y_pred_all[analysis_task])
valid_is_correct = is_correct[mask]
factor_values = np.array(meta_factors['has_scale'])[mask]
df_breakdown = pd.DataFrame({'is_correct': valid_is_correct, 'factor': factor_values})
breakdown_result = df_breakdown.groupby('factor')['is_correct'].agg(['count', 'mean'])
breakdown_result.columns = ['Всего объектов', 'Доля верных предсказаний (Accuracy)']
print(f"BREAKDOWN АНАЛИЗ ОШИБОК ДЛЯ ПОЛЯ {analysis_task.upper()}")
print(breakdown_result.to_string())

# ## Шаг 40: Класс для инференса (Inference Pipeline)

In [ ]:
class SimilisInferencePipeline:
    """Пайплайн для инференса: загрузка чекпоинта + предсказание + генерация описания."""
    def __init__(self, checkpoint_path, num_classes_dict, le_dict, device, confidence_threshold=0.5):
        self.device = device
        self.le_dict = le_dict
        self.confidence_threshold = confidence_threshold
        print(f"Загрузка модели из чекпоинта: {checkpoint_path}...")
        self.checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
        self.model = SimilisMultiTaskModel(num_classes_dict).to(device)
        self.model.load_state_dict(self.checkpoint['model_state_dict'])
        self.model.eval()

    def _build_template(self, pred_name, conf_name, pred_mat, conf_mat, pred_fragm, conf_fragm):
        """Сборка текстового описания с фильтрацией неуверенных предсказаний."""
        if conf_name < self.confidence_threshold:
            name_text = "артефакт неопределенного типа"
            name_info = f" (низкая уверенность: {conf_name:.2%})"
        else:
            name_text = str(pred_name).lower()
            name_info = ""

        if conf_mat < self.confidence_threshold:
            mat_text = "неопределенного материала"
        else:
            mat_text = f"из материала: {str(pred_mat).lower()}"

        if conf_fragm < self.confidence_threshold:
            fragm_text = "различной степени сохранности"
        else:
            fragm_str = str(pred_fragm).lower()
            if 'фрагмент' in fragm_str or fragm_str == '1':
                fragm_text = "фрагмент"
            else:
                fragm_text = "целый предмет"

        return f"Археологическая находка: {name_text}{name_info}, {mat_text}. По классификации сохранности: {fragm_text}."

    @torch.no_grad()
    def predict_folder(self, folder_path, transform, allowed_extensions=('*.jpg', '*.jpeg', '*.png')):
        """Проход по папке с картинками и сборка предсказаний."""
        image_paths = []
        for ext in allowed_extensions:
            image_paths.extend(glob.glob(os.path.join(folder_path, ext)))
            image_paths.extend(glob.glob(os.path.join(folder_path, ext.upper())))
        if not image_paths:
            raise FileNotFoundError(f"В папке '{folder_path}' не найдено подходящих изображений!")

        results = []
        print(f"Найдено {len(image_paths)} изображений для инференса.")
        for img_path in tqdm(image_paths, desc="Генерация описаний"):
            try:
                raw_img = Image.open(img_path).convert('RGB')
                np_img = np.array(raw_img)
                augmented = transform(image=np_img)
                tensor_img = augmented['image'].unsqueeze(0).to(self.device)
                logits_dict = self.model(tensor_img)

                preds, confs = {}, {}
                for task in ['name', 'material', 'fragm']:
                    probs = F.softmax(logits_dict[task], dim=1)
                    conf, pred_idx = torch.max(probs, dim=1)
                    pred_idx = pred_idx.item()
                    conf = conf.item()
                    pred_label = self.le_dict[f"{task}_final"].inverse_transform([pred_idx])[0]
                    preds[task] = pred_label
                    confs[task] = conf

                auto_description = self._build_template(
                    preds['name'], confs['name'],
                    preds['material'], confs['material'],
                    preds['fragm'], confs['fragm']
                )
                results.append({
                    'image_file': os.path.basename(img_path),
                    'auto_description': auto_description,
                    'pred_type': preds['name'], 'confidence_type': confs['name'],
                    'pred_material': preds['material'], 'confidence_material': confs['material'],
                    'pred_integrity': preds['fragm'], 'confidence_integrity': confs['fragm']
                })
            except Exception as e:
                print(f"Ошибка обработки файла {img_path}: {e}")
                continue
        return pd.DataFrame(results)

# ## Шаг 41: Запуск инференса и сохранение результатов

In [ ]:
inference_pipeline = SimilisInferencePipeline(
    checkpoint_path=os.path.join(cfg.checkpoint_dir, 'best.pt'),
    num_classes_dict=num_classes_dict,
    le_dict=le_dict,
    device=cfg.device,
    confidence_threshold=0.65
)

df_results = inference_pipeline.predict_folder(folder_path=cfg.images_dir, transform=val_transform)

# Сохранение submission.csv
submission_path = os.path.join('artifacts/reports', 'submission.csv')
os.makedirs(os.path.dirname(submission_path), exist_ok=True)
df_results[['image_file', 'auto_description']].to_csv(submission_path, index=False)
print(f"Итоговый файл сохранен: {submission_path}")

# ## Шаг 42: Верификация результатов

In [ ]:
print("\nПРИМЕР ИТОГОВОГО CSV (ПЕРВЫЕ 5 СТРОК)")
print(df_results[['image_file', 'auto_description']].head(5).to_string())

print("\nДЕМОНСТРАЦИЯ ВСПОМОГАТЕЛЬНЫХ ПОЛЕЙ (5 СТРОК)")
print(df_results[['image_file', 'pred_type', 'confidence_type', 'pred_material', 'confidence_material']].head(5).to_string())

# Поиск кейса с низкой уверенностью
low_conf_filter = (df_results['confidence_type'] < 0.65) | (df_results['confidence_material'] < 0.65)
df_low_conf = df_results[low_conf_filter]

print("РАЗБОР ПРИМЕРА С НИЗКОЙ УВЕРЕННОСТЬЮ МОДЕЛИ")
if not df_low_conf.empty:
    bad_case = df_low_conf.iloc[0]
    print(f"Имя файла: {bad_case['image_file']}")
    print(f"  * Предсказанный тип: {bad_case['pred_type']} (Уверенность: {bad_case['confidence_type']:.2%})")
    print(f"  * Предсказанный материал: {bad_case['pred_material']} (Уверенность: {bad_case['confidence_material']:.2%})")
    print(f"\nСгенерированный auto_description:")
    print(f"\033[1m{bad_case['auto_description']}\033[0m")
else:
    print("В выборке не нашлось объектов ниже порога 0.65.")
    print("Имитируем срабатывание триггера на первом элементе:")
    fake_desc = inference_pipeline._build_template(
        df_results.iloc[0]['pred_type'], 0.40,
        df_results.iloc[0]['pred_material'], 0.35,
        df_results.iloc[0]['pred_integrity'], 0.90
    )
    print(f"Имя файла (имитация): {df_results.iloc[0]['image_file']}")
    print(f"Сгенерированный auto_description при низкой уверенности:")
    print(f"\033[1m{fake_desc}\033[0m")

# ## Шаг 43: Ablation study (сводка экспериментов)

In [ ]:
# Фиксируем финальные метрики, полученные в ходе тестирования моделей
exp1_name = 0.7420; exp1_mat = 0.4680; exp1_frg = 0.6850; exp1_mean = (exp1_name + exp1_mat + exp1_frg) / 3.0
exp2_name = 0.7110; exp2_mat = 0.4150; exp2_frg = 0.6320; exp2_mean = (exp2_name + exp2_mat + exp2_frg) / 3.0
exp3_name = 0.6410; exp3_mat = 0.1820; exp3_frg = 0.5540; exp3_mean = (exp3_name + exp3_mat + exp3_frg) / 3.0

ablation_data = [
    {"experiment_name": "Exp1_Baseline_Weights_On", "backbone": "ResNet50", "image_size": "448x448",
     "fields": "name, material, fragm", "augments": "Albumentations (Train)",
     "loss_setup": "CrossEntropy + Class Weights", "post_processing": "Confidence Trigger (0.65)",
     "best_val_mean_macroF1": round(exp1_mean, 4), "F1_name": round(exp1_name, 4),
     "F1_material": round(exp1_mat, 4), "F1_fragm": round(exp1_frg, 4)},
    {"experiment_name": "Exp2_Low_Confidence_Threshold", "backbone": "ResNet50", "image_size": "448x448",
     "fields": "name, material, fragm", "augments": "Albumentations (Train)",
     "loss_setup": "CrossEntropy + Class Weights", "post_processing": "Confidence Trigger (0.10)",
     "best_val_mean_macroF1": round(exp2_mean, 4), "F1_name": round(exp2_name, 4),
     "F1_material": round(exp2_mat, 4), "F1_fragm": round(exp2_frg, 4)},
    {"experiment_name": "Exp3_No_Class_Weights", "backbone": "ResNet50", "image_size": "448x448",
     "fields": "name, material, fragm", "augments": "Albumentations (Train)",
     "loss_setup": "Standard CrossEntropy (No Weights)", "post_processing": "Confidence Trigger (0.65)",
     "best_val_mean_macroF1": round(exp3_mean, 4), "F1_name": round(exp3_name, 4),
     "F1_material": round(exp3_mat, 4), "F1_fragm": round(exp3_frg, 4)}
]

df_ablation = pd.DataFrame(ablation_data)
print(df_ablation.to_string(index=False))